# Scrape news from sites mine companies

### Load required librarys

In [2]:
import os
import time
import json
import urllib.parse
import requests
from io import BytesIO
from pypdf import PdfReader

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [3]:
# Coockies block
# Handle cookies
try:
    cookie_btn = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "button#onetrust-accept-btn-handler"))
    )
    cookie_btn.click()
    time.sleep(1)
except:
    pass

## Amerigo Resources Ltd

Scarper for two years, with pdf files as articles

### 2024

In [ ]:
# -------------------------
# PDF HANDLING FUNCTIONS
# -------------------------

def is_pdf_url(url: str) -> bool:
    """Detect if the URL points to a PDF, even if it doesn't end with .pdf."""
    parsed = urllib.parse.urlparse(url)

    # Case 1: direct .pdf path
    if parsed.path.lower().endswith(".pdf"):
        return True

    # Case 2: viewer?file=...pdf
    qs = urllib.parse.parse_qs(parsed.query)
    if "file" in qs:
        file_url = qs["file"][0]
        if file_url.lower().endswith(".pdf"):
            return True

    # Case 3: check content-type from server
    try:
        head = requests.head(url, allow_redirects=True, timeout=5)
        if head.headers.get("Content-Type", "").startswith("application/pdf"):
            return True
    except:
        pass

    return False


def extract_pdf_text_from_url(url: str) -> str:
    """Download PDF with proper browser headers."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "application/pdf",
        "Referer": "https://www.amerigoresources.com/",
        "Connection": "keep-alive",
    }

    response = requests.get(url, headers=headers, allow_redirects=True)
    response.raise_for_status()

    pdf_reader = PdfReader(BytesIO(response.content))
    pages = [page.extract_text() or "" for page in pdf_reader.pages]
    return "\n".join(pages)



# -------------------------
# MAIN SCRAPER
# -------------------------

def scrape_arg_news():
    driver = webdriver.Chrome()
    driver.get("https://www.amerigoresources.com/investors/news/2024/")
    
    wait = WebDriverWait(driver, 10)
    articles_data = []
    
    try:
        # Get links + dates
        article_links = driver.find_elements(By.CSS_SELECTOR, ".listing-item__title")
        article_dates_elements = driver.find_elements(By.CSS_SELECTOR, ".listing-item__date")

        article_urls = [a.get_attribute('href') for a in article_links]
        article_dates = [d.text for d in article_dates_elements]

        print(f"Found {len(article_urls)} articles")

        # Loop through articles
        for idx, url in enumerate(article_urls):
            print(f"\nProcessing ({idx+1}/{len(article_urls)}): {url}")

            # -------------------------------------------
            # 👉 PDF HANDLING
            # -------------------------------------------
            if is_pdf_url(url):
                try:
                    print("  PDF detected → downloading...")
                    pdf_text = extract_pdf_text_from_url(url)

                    # Use filename as title fallback
                    filename = os.path.basename(url).split("?")[0]
                    title = filename.replace(".pdf", "").replace("-", " ").replace("_", " ")

                    articles_data.append({
                        "title": article_links[idx].text,
                        "date": article_dates[idx],
                        "content": pdf_text,
                        "url": url
                    })

                    print("  ✓ PDF scraped successfully")

                except Exception as e:
                    print(f"  ERROR extracting PDF: {e}")

                continue  # Skip to next item (no HTML scraping needed)


        # -------------------------
        # SAVE TO JSON
        # -------------------------
        os.makedirs("./Data", exist_ok=True)

        existing_data = []
        if os.path.exists("./Data/arg_news.json"):
            try:
                with open("./Data/arg_news.json", "r", encoding="utf-8") as f:
                    existing_data = json.load(f)
                print(f"\nLoaded {len(existing_data)} existing items")
            except:
                print("Warning: Could not read existing JSON.")

        existing_urls = {item["url"] for item in existing_data}

        new_items = [
            item for item in articles_data
            if item["url"] not in existing_urls
        ]

        combined = existing_data + new_items

        with open("./Data/arg_news.json", "w", encoding="utf-8") as f:
            json.dump(combined, f, indent=2, ensure_ascii=False)

        print(f"\nAdded {len(new_items)} new items. Total: {len(combined)}")

    finally:
        driver.quit()

    return articles_data


if __name__ == "__main__":
    articles = scrape_arg_news()
    print(f"\nScraped {len(articles)} items")


Found 15 articles

Processing (1/15): https://amerigoresources.com/_resources/news/nr-20241128.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (2/15): https://amerigoresources.com/_resources/news/nr-20241030.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (3/15): https://amerigoresources.com/_resources/news/nr-20241008.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (4/15): https://amerigoresources.com/_resources/news/nr-20240731.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (5/15): https://amerigoresources.com/_resources/news/nr-20240710.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (6/15): https://amerigoresources.com/_resources/news/nr-20240709.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (7/15): https://amerigoresources.com/_resources/news/nr-20240702.pdf
  PDF detected → downloading...
  ✓ PDF scraped s

### 2025

In [8]:
# -------------------------
# PDF HANDLING FUNCTIONS
# -------------------------

def is_pdf_url(url: str) -> bool:
    """Detect if the URL points to a PDF, even if it doesn't end with .pdf."""
    parsed = urllib.parse.urlparse(url)

    # Case 1: direct .pdf path
    if parsed.path.lower().endswith(".pdf"):
        return True

    # Case 2: viewer?file=...pdf
    qs = urllib.parse.parse_qs(parsed.query)
    if "file" in qs:
        file_url = qs["file"][0]
        if file_url.lower().endswith(".pdf"):
            return True

    # Case 3: check content-type from server
    try:
        head = requests.head(url, allow_redirects=True, timeout=5)
        if head.headers.get("Content-Type", "").startswith("application/pdf"):
            return True
    except:
        pass

    return False


def extract_pdf_text_from_url(url: str) -> str:
    """Download PDF with proper browser headers."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "application/pdf",
        "Referer": "https://www.amerigoresources.com/",
        "Connection": "keep-alive",
    }

    response = requests.get(url, headers=headers, allow_redirects=True)
    response.raise_for_status()

    pdf_reader = PdfReader(BytesIO(response.content))
    pages = [page.extract_text() or "" for page in pdf_reader.pages]
    return "\n".join(pages)



# -------------------------
# MAIN SCRAPER
# -------------------------

def scrape_arg_news():
    driver = webdriver.Chrome()
    driver.get("https://www.amerigoresources.com/investors/news/2025/")
    
    wait = WebDriverWait(driver, 10)
    articles_data = []
    
    try:
        # Get links + dates
        article_links = driver.find_elements(By.CSS_SELECTOR, ".listing-item__title")
        article_dates_elements = driver.find_elements(By.CSS_SELECTOR, ".listing-item__date")

        article_urls = [a.get_attribute('href') for a in article_links]
        article_dates = [d.text for d in article_dates_elements]

        print(f"Found {len(article_urls)} articles")

        # Loop through articles
        for idx, url in enumerate(article_urls):
            print(f"\nProcessing ({idx+1}/{len(article_urls)}): {url}")

            # -------------------------------------------
            # 👉 PDF HANDLING
            # -------------------------------------------
            if is_pdf_url(url):
                try:
                    print("  PDF detected → downloading...")
                    pdf_text = extract_pdf_text_from_url(url)

                    # Use filename as title fallback
                    filename = os.path.basename(url).split("?")[0]
                    title = filename.replace(".pdf", "").replace("-", " ").replace("_", " ")

                    articles_data.append({
                        "title": article_links[idx].text,
                        "date": article_dates[idx],
                        "content": pdf_text,
                        "url": url,
                        "type": "pdf"
                    })

                    print("  ✓ PDF scraped successfully")

                except Exception as e:
                    print(f"  ERROR extracting PDF: {e}")

                continue  # Skip to next item (no HTML scraping needed)


        # -------------------------
        # SAVE TO JSON
        # -------------------------
        os.makedirs("./Data", exist_ok=True)

        existing_data = []
        if os.path.exists("./Data/arg_news.json"):
            try:
                with open("./Data/arg_news.json", "r", encoding="utf-8") as f:
                    existing_data = json.load(f)
                print(f"\nLoaded {len(existing_data)} existing items")
            except:
                print("Warning: Could not read existing JSON.")

        existing_urls = {item["url"] for item in existing_data}

        new_items = [
            item for item in articles_data
            if item["url"] not in existing_urls
        ]

        combined = existing_data + new_items

        with open("./Data/arg_news.json", "w", encoding="utf-8") as f:
            json.dump(combined, f, indent=2, ensure_ascii=False)

        print(f"\nAdded {len(new_items)} new items. Total: {len(combined)}")

    finally:
        driver.quit()

    return articles_data


if __name__ == "__main__":
    articles = scrape_arg_news()
    print(f"\nScraped {len(articles)} items")


Found 17 articles

Processing (1/17): https://amerigoresources.com/_resources/news/nr-20251202.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (2/17): https://amerigoresources.com/_resources/news/nr-20251128.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (3/17): https://amerigoresources.com/_resources/news/nr-20251029.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (4/17): https://amerigoresources.com/_resources/news/nr-20251023.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (5/17): https://amerigoresources.com/_resources/news/nr-20251008.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (6/17): https://amerigoresources.com/_resources/news/nr-20250813.pdf
  PDF detected → downloading...
  ✓ PDF scraped successfully

Processing (7/17): https://amerigoresources.com/_resources/news/nr-20250806.pdf
  PDF detected → downloading...
  ✓ PDF scraped s

## Aya Gold & Silver Inc

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time
import json
import os

scraper with extra button to go to article (also with date check)

In [16]:

def scrape_aya_news():
    driver = webdriver.Chrome()
    driver.get("https://ayagoldsilver.com/press-releases/")
    
    wait = WebDriverWait(driver, 10)
    articles_data = []

    try:
        # ==============================
        # 1. GET TITLES, LINKS, DATES (DAY/MONTH/YEAR)
        # ==============================
        titles = driver.find_elements(By.CSS_SELECTOR, "h4")
        links = driver.find_elements(By.CSS_SELECTOR, ".mt-auto")

        days = driver.find_elements(By.CSS_SELECTOR, ".day")
        months = driver.find_elements(By.CSS_SELECTOR, ".month")
        years = driver.find_elements(By.CSS_SELECTOR, ".year")

        print("Total articles found on page:", len(titles))

        # Build structured metadata
        metadata = []
        for t, l, d, m, y in zip(titles, links, days, months, years):
            full_date = f"{d.text.strip()} {m.text.strip()} {y.text.strip()}"
            year_value = y.text.strip()

            # Filter YEARS 2024–2025
            if year_value not in ["2024", "2025"]:
                continue

            metadata.append({
                "list_title": t.text.strip(),
                "url": l.get_attribute("href"),
                "date": full_date
            })

        print("Filtered to:", len(metadata), "articles from 2024–2025")

        # ==============================
        # 2. SCRAPE EACH ARTICLE PAGE
        # ==============================
        for item in metadata:
            url = item["url"]
            driver.get(url)
            time.sleep(1)

            try:
                page_title = driver.find_element(By.CSS_SELECTOR, "h1").text.strip()

                content_blocks = driver.find_elements(By.CSS_SELECTOR, "li, .col-md-7 p")
                content = "\n".join(c.text.strip() for c in content_blocks if c.text.strip())

                articles_data.append({
                    "title": page_title,
                    "date": item["date"],
                    "content": content,
                    "url": url
                })

                print("Scraped:", page_title)

            except Exception as e:
                print("Error scraping article:", url, e)

        # ==============================
        # 3. SAVE TO JSON (avoid duplicates)
        # ==============================
        os.makedirs("./Data", exist_ok=True)
        json_path = "./Data/aya_news.json"

        if os.path.exists(json_path):
            try:
                with open(json_path, "r", encoding="utf-8") as f:
                    existing = json.load(f)
            except:
                existing = []
        else:
            existing = []

        existing_urls = {a["url"] for a in existing}
        new_articles = [a for a in articles_data if a["url"] not in existing_urls]

        combined = existing + new_articles

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(combined, f, indent=2, ensure_ascii=False)

        print(f"Added {len(new_articles)} new articles. Total now: {len(combined)}")

    finally:
        driver.quit()

    return articles_data


if __name__ == "__main__":
    data = scrape_aya_news()
    print(f"Scraped {len(data)} articles")


Total articles found on page: 407
Filtered to: 131 articles from 2024–2025
Scraped: Aya Gold & Silver Unlocks Near-Term Value and Advances Environmental Remediation with Sale of Precious Metals from Legacy Boumadine Stockpile
Scraped: Aya Gold & Silver Reports Record Revenue and Net Income for Q3-2025 on Record Production
Scraped: Aya Or & Argent annonce des teneurs élevées en argent à Zgounder et découvre une nouvelle zone minéralisée au nord
Scraped: Aya Gold & Silver Reports Strong Q2-2025 Results
Scraped: Aya Gold & Silver Receives US$8 Million in Compensation Related to Zgounder Expansion
Scraped: Aya Or & Argent reçoit 8 millions de dollars américains enrémunération liée à l’expansion de la mine Zgounder
Scraped: Aya Gold & Silver Reports Q2-2025 and Mid-July Production Results
Scraped: Aya Gold & Silver Reports High-Grade Boumadine Exploration Results, Expands Imariren to 1km and Identifies New High-Grade Asirem Zone to the West
Scraped: Aya Or & Argent inc. publie des résultats

## Centerra Gold Inc

Scraper with button to go to second page, and no year in the url (year button)

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import os

### 2024

In [ ]:

def scrape_cg_news():
    driver = webdriver.Chrome()
    driver.get("https://www.centerragold.com/investor-news/default.aspx")
    wait = WebDriverWait(driver, 15)

    # Handle cookies
    try:
        cookie_btn = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button#onetrust-accept-btn-handler"))
        )
        cookie_btn.click()
        time.sleep(1)
    except:
        pass

    # 🔥🔥 CHANGE THIS LINE ONLY to switch year
    YEAR_TO_SCRAPE = "2024"     # <-- Change "2024" to "2025" when needed

    # -------------------------------------------------------------

    # Select year
    dropdown = Select(wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#newsYear"))))
    dropdown.select_by_visible_text(YEAR_TO_SCRAPE)
    time.sleep(2)

    def extract_list_articles():
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".module_headline-link")))
        titles = driver.find_elements(By.CSS_SELECTOR, ".module_headline-link")
        dates = driver.find_elements(By.CSS_SELECTOR, ".module_date-time")

        urls  = [t.get_attribute("href") for t in titles]
        ttext = [t.text.strip() for t in titles]
        dtext = [d.text.strip() for d in dates]

        return list(zip(ttext, dtext, urls))

    articles = []

    # Page 1
    articles.extend(extract_list_articles())

    # Page 2 (if exists)
    try:
        next_page = wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR, ".pager_button--next")
            )
        ).click()
            
        time.sleep(2)
        articles.extend(extract_list_articles())
        print("second page found")
    except:
        print("No second page found")

    # Scrape article pages ------------------------
    scraped = []

    for title, date, url in articles:
        print("Scraping:", title)
        try:
            driver.get(url)
            wait.until(EC.presence_of_element_located(
                (By.CSS_SELECTOR, "#_ctrl0_ctl61_lblModuleDetailHeadline")
            ))

            full_title = driver.find_element(
                By.CSS_SELECTOR, "#_ctrl0_ctl61_lblModuleDetailHeadline"
            ).text

            paragraphs = driver.find_elements(
                By.CSS_SELECTOR, "#_ctrl0_ctl61_divModuleContainer p, #_ctrl0_ctl61_divModuleContainer em"
            )

            content = "\n\n".join([p.text.strip() for p in paragraphs if p.text.strip()])

            scraped.append({
                "title": full_title,
                "date": date,
                "content": content,
                "url": url
            })

        except Exception as e:
            print("Error:", e)

    # Save JSON -----------------------------------------
    os.makedirs("./Data", exist_ok=True)
    output_path = f"./Data/cg_news.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(scraped, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(scraped)} articles to {output_path}")
    driver.quit()

    return scraped


if __name__ == "__main__":
    scrape_cg_news()


Scraping: Centerra Gold Announces TSX Acceptance of Renewed Normal Course Issuer Bid
Scraping: Centerra Gold Announces Quarterly Dividend of C$0.07 per Common Share
Scraping: Centerra Gold Reports Third Quarter 2024 Results; Consistent Operating Performance Drives Continued Strong Cash Flow From Operations
Scraping: Centerra Gold Provides Notice of Third Quarter 2024 Results Release and Conference Call
Scraping: Centerra Gold Announces Thompson Creek Feasibility Study Results and Strategic Plan for US Molybdenum Operations, Including a Restart of the Thompson Creek Mine and Ramp-Up of Langeloth
Scraping: Centerra Gold Provides Notice of Release and Conference Call for the Thompson Creek Feasibility Study Results and Strategic Plan for the US Molybdenum Operations
Scraping: Centerra Gold Announces Quarterly Dividend of C$0.07 per Common Share
Scraping: Centerra Gold Reports Second Quarter 2024 Results; Consistent Operating Performance; Continued Return of Capital to Shareholders
Scrapin

### 2025

In [ ]:

def scrape_cg_news():
    driver = webdriver.Chrome()
    driver.get("https://www.centerragold.com/investor-news/default.aspx")
    wait = WebDriverWait(driver, 15)

    # Handle cookies
    try:
        cookie_btn = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button#onetrust-accept-btn-handler"))
        )
        cookie_btn.click()
        time.sleep(1)
    except:
        pass

    # 🔥🔥 CHANGE THIS LINE ONLY to switch year
    YEAR_TO_SCRAPE = "2025"     # <-- Change "2024" to "2025" when needed

    # -------------------------------------------------------------

    # Select year
    dropdown = Select(wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#newsYear"))))
    dropdown.select_by_visible_text(YEAR_TO_SCRAPE)
    time.sleep(2)

    def extract_list_articles():
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".module_headline-link")))
        titles = driver.find_elements(By.CSS_SELECTOR, ".module_headline-link")
        dates = driver.find_elements(By.CSS_SELECTOR, ".module_date-time")

        urls  = [t.get_attribute("href") for t in titles]
        ttext = [t.text.strip() for t in titles]
        dtext = [d.text.strip() for d in dates]

        return list(zip(ttext, dtext, urls))

    articles = []

    # Page 1
    articles.extend(extract_list_articles())

    # Page 2 (if exists)
    try:
        next_page = wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR, ".pager_button--next")
            )
        ).click()
        
        time.sleep(2)
        articles.extend(extract_list_articles())
        print("Second page found")
    except:
        print("No second page found")

    # Scrape article pages ------------------------
    scraped = []

    for title, date, url in articles:
        print("Scraping:", title)
        try:
            driver.get(url)
            wait.until(EC.presence_of_element_located(
                (By.CSS_SELECTOR, "#_ctrl0_ctl61_lblModuleDetailHeadline")
            ))

            full_title = driver.find_element(
                By.CSS_SELECTOR, "#_ctrl0_ctl61_lblModuleDetailHeadline"
            ).text

            paragraphs = driver.find_elements(
                By.CSS_SELECTOR, "#_ctrl0_ctl61_divModuleContainer p, #_ctrl0_ctl61_divModuleContainer em"
            )

            content = "\n\n".join([p.text.strip() for p in paragraphs if p.text.strip()])

            scraped.append({
                "title": full_title,
                "date": date,
                "content": content,
                "url": url
            })

        except Exception as e:
            print("Error:", e)

    # Save JSON -----------------------------------------
    os.makedirs("./Data", exist_ok=True)
    output_path = f"./Data/cg_news.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(scraped, f, indent=2, ensure_ascii=False)

    print(f"\nSaved {len(scraped)} articles to {output_path}")
    driver.quit()

    return scraped


if __name__ == "__main__":
    scrape_cg_news()


No second page found
Scraping: Centerra Gold Announces TSX Acceptance of Renewed Normal Course Issuer Bid
Scraping: Centerra Gold Announces Quarterly Dividend of C$0.07 per Common Share
Scraping: Centerra Gold Reports Third Quarter 2025 Results; Strong Production at Öksüt and Higher Metal Prices Boosted Cash Balance to $562 Million; Financial Strength Drives Centerra’s Self-Funded Growth Strategy; Board Chair Transition Underscores Continued Leadership Strength
Scraping: Centerra Gold Files Technical Report for the Mount Milligan Mine
Scraping: Centerra Gold Provides Notice of Third Quarter 2025 Results and Conference Call
Scraping: Centerra Gold’s Mount Milligan PFS Outlines Mine Life to 2045, Delivering Growth with a Fully Funded, Disciplined $186 Million Growth Capital Plan
Scraping: Centerra Gold Announces Quarterly Dividend of C$0.07 per Common Share
Scraping: Centerra Gold Reports Second Quarter 2025 Results; Reinforced Balance Sheet Strength with Strong Operational Cash Flow Per

## Canadian gold Grorp

In [30]:
"""
A web scraper for Canadian Gold Corp news articles using Selenium.
Saves scraped articles to a JSON file, appending new articles while avoiding duplicates.
SelectorGadget: https://selectorgadget.com/
"""

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time
import json
import os

def scrape_cgc_news():
    # Initialize the driver
    driver = webdriver.Chrome()  # or webdriver.Firefox()
    driver.get("https://canadiangoldcorp.com/news-releases-2025/")
    #driver.get("https://canadiangoldcorp.com/news-2024/")
    
    wait = WebDriverWait(driver, 10)
    articles_data = []
    
    try:
        # You can find the CSS selectors by using SelectorGadget or inspecting the webpage
        article_links = driver.find_elements(By.CSS_SELECTOR, ".wp-show-posts-entry-title a")
        article_dates_elements = driver.find_elements(By.CSS_SELECTOR, ".published")
        article_urls = [link.get_attribute('href') for link in article_links]
        article_dates = [date.text for date in article_dates_elements]
        
        print(f"Found {len(article_urls)} articles to scrape")

        # Navigate to each article URL and extract data
        for idx, url in enumerate(article_urls):
            driver.get(url)
            time.sleep(2)  # Wait for page load
            
            try:
                title = driver.find_element(By.CSS_SELECTOR, "#primary .ut-flowtyped").text
                content = driver.find_element(By.CSS_SELECTOR, ".category-78").text # for 2025
                #content = driver.find_element(By.CSS_SELECTOR, ".category-77").text # for 2024
                
                articles_data.append({
                    'title': title,
                    'date': article_dates[idx] if idx < len(article_dates) else "Unknown",
                    'content': content,
                    'url': url
                })
                
                print(f"Scraped: {title}")
            except Exception as e:
                print(f"Error scraping {url}: {e}")
        
        # Load existing data if file exists, otherwise start with empty list
        existing_data = []
        if os.path.exists('./Data/cgc_news.json'):
            try:
                with open('./Data/cgc_news.json', 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
                print(f"Loaded {len(existing_data)} existing articles")
            except json.JSONDecodeError:
                print("Warning: Existing JSON file was invalid, starting fresh")
        
        # Combine existing and new data (avoiding duplicates based on URL)
        existing_urls = {article['url'] for article in existing_data}
        new_articles = [article for article in articles_data if article['url'] not in existing_urls]
        
        combined_data = existing_data + new_articles
        
        # Save combined data to JSON
        with open('./Data/cgc_news.json', 'w', encoding='utf-8') as f:
            json.dump(combined_data, f, indent=2, ensure_ascii=False)
        
        print(f"Added {len(new_articles)} new articles. Total: {len(combined_data)} articles")
                
    finally:
        driver.quit()
    
    return articles_data

if __name__ == "__main__":
    articles = scrape_cgc_news()
    print(f"Scraped {len(articles)} articles")

python(51610) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51631) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Found 20 articles to scrape
Scraped: Canadian Gold Corp. Tartan Mine “Western Flank” Step-Out Drilling Continues To Deliver
Scraped: Manitoba Mineral Development Fund Grants Canadian Gold Corp. $300,000 to Expand Exploration Program
Scraped: Canadian Gold Corp. Announces Sending of Meeting Materials for Proposed Plan of Arrangement
Scraped: McEwen Inc. and Canadian Gold Corp. Announce Arrangement Agreement
Scraped: McEwen Inc. and Canadian Gold Corp. Announce Letter of Intent
Scraped: Canadian Gold Corp. Field Exploration Underway at Hammond Reef South & Tartan West
Scraped: Canadian Gold Corp. Closes Over-Subscribed Private Placement for New Hammond Reef South Program
Scraped: Canadian Gold Corp. Lifting of Evacuation Order in Flin Flon, Manitoba
Scraped: Canadian Gold Corp. Private Placement Offering for New Hammond Reef South Program to Follow up 2024 Surface Samples that Returned 35.4 g/t Gold
Scraped: Canadian Gold Corp. Intersects Two Separate High-Grade Zones Returning 12.9 gpt 

## Chesapeake Gold Corp

Meerdere pages /x in de link aanpassen en jaar ook juist zetten

In [1]:
import os
import time
import json
import urllib.parse
import requests
from io import BytesIO
from pypdf import PdfReader

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### 2024 en 2025

In [28]:
def scrape_chesapeake_news():
    driver = webdriver.Chrome()

    # 🔥 CHANGE PAGE NUMBER MANUALLY (example: /2, /3, /4, /5)
    driver.get("https://chesapeakegold.com/news/4")

    wait = WebDriverWait(driver, 15)

    # 🔥 CHANGE THIS LINE ONLY FOR YEAR
    YEAR_TO_SCRAPE = "2024"

    # ------------------------------------------------------------------
    def extract_list_articles():
        """Extract article titles, dates, URLs from current list page."""
        wait.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, ".elementor-post__title a")
            )
        )

        titles = driver.find_elements(By.CSS_SELECTOR, ".elementor-post__title a")
        dates  = driver.find_elements(By.CSS_SELECTOR, ".elementor-post-date")

        urls  = [t.get_attribute("href") for t in titles]
        ttext = [t.text.strip() for t in titles]
        dtext = [d.text.strip() for d in dates]

        # Only return articles from chosen year
        filtered = [
            (title, date, url)
            for title, date, url in zip(ttext, dtext, urls)
            if YEAR_TO_SCRAPE in date
        ]

        return filtered

    # -------------------------
    # EXTRACT CURRENT PAGE
    # -------------------------
    articles_data = extract_list_articles()
    print(f"Found {len(articles_data)} articles for year {YEAR_TO_SCRAPE}")

    # -------------------------
    # SCRAPE ARTICLE PAGES
    # -------------------------
    scraped_articles = []

    for idx, (title, date, url) in enumerate(articles_data):
        print(f"Scraping ({idx+1}/{len(articles_data)}): {title}")

        try:
            driver.get(url)
            wait.until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, ".elementor-size-default")
                )
            )

            full_title = driver.find_element(
                By.CSS_SELECTOR, ".elementor-size-default"
            ).text.strip()

            paragraphs = driver.find_elements(
                By.CSS_SELECTOR, ".elementor-widget-container > p"
            )
            content = "\n\n".join(
                [p.text.strip() for p in paragraphs if p.text.strip()]
            )

            scraped_articles.append({
                "title": full_title,
                "date": date,
                "content": content,
                "url": url
            })

        except Exception as e:
            print("Error:", e)

    # -------------------------
    # JSON MERGE + DUPLICATE FILTER
    # -------------------------
    os.makedirs("./Data", exist_ok=True)
    json_path = './Data/Chesapeake_news.json'

    # Load existing
    existing_data = []
    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                existing_data = json.load(f)
            print(f"Loaded {len(existing_data)} existing articles")
        except json.JSONDecodeError:
            print("Warning: Existing JSON file invalid. Starting fresh.")

    # Deduplicate
    existing_urls = {article['url'] for article in existing_data}
    new_articles = [a for a in scraped_articles if a['url'] not in existing_urls]

    combined_data = existing_data + new_articles

    # Save
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(combined_data, f, indent=2, ensure_ascii=False)

    print(f"Added {len(new_articles)} new articles. Total now: {len(combined_data)}")

    driver.quit()
    return scraped_articles


if __name__ == "__main__":
    scrape_chesapeake_news()


python(51219) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51240) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Found 1 articles for year 2024
Scraping (1/1): CHESAPEAKE GOLD COMPLETES VERTICAL SHORT-FORM AMALGAMATION WITH WHOLLY-OWNED SUBSIDIARY
Loaded 21 existing articles
Added 1 new articles. Total now: 22


## District Metals Corp

### Main Scraper functions

In [2]:
import os
import time
import json
from datetime import datetime

from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [4]:
# -------------------------------------------------------------------
# DRIVER & UTILITIES
# -------------------------------------------------------------------

def create_driver(headless: bool = False) -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    # Perf tweak: block some heavy content (safe to remove if it breaks anything)
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )

# -------------------------------------------------------------------
# Click Button
# -------------------------------------------------------------------
def click_button(driver, CSS_button):
    wait = WebDriverWait(driver, 10)
    button = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, CSS_button))
    )
    button.click()

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def collect_header_info_page(driver, CSS_header: None, BASE_URL, year_list, CSS_date, CSS_title, CSS_url: None) -> list[dict]:
    collected_headers = []
    wait = WebDriverWait(driver, 10)

    # Go to the base page once
    driver.get(BASE_URL)

    time.sleep(2)

    for year in year_list:
        year_str = str(year)

        BASE_URL = f'{BASE_URL}#{year_str}'

        # 1) Wait until a filter link with this text exists (NO click here)
        wait_script = """
            return [...document.querySelectorAll('a.filter')]
                   .some(a => a.textContent.trim() === arguments[0]);
        """
        wait.until(lambda d: d.execute_script(wait_script, year_str))

        # 2) Click that filter link via JS
        click_script = """
            let el = [...document.querySelectorAll('a.filter')]
                     .find(a => a.textContent.trim() === arguments[0]);
            if (el) el.click();
        """
        driver.execute_script(click_script, year_str)

        # 3) Optional: small wait so the filtered list can update (tune selector to your page)
        #    e.g. wait for at least one post in that year to be visible
        # wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "your-post-selector")))

        # 4) Collect headers for the CURRENT DOM, using your existing helper.
        #    We pass CSS_header=None so collect_header_info uses the "title has the URL" branch.
        #    We don't mutate BASE_URL; if your collect_header_info does driver.get(BASE_URL),
        #    you can pass base_no_hash or base_no_hash + f"#{year_str}" depending on your setup.
        headers = collect_header_info(
            driver,
            BASE_URL,   
            CSS_header,
            CSS_date,
            CSS_title,
            None,
            CSS_url
        )

        collected_headers.extend(headers)

    return collected_headers


def collect_header_info(driver, BASE_URL, CSS_header, CSS_date, CSS_title, CSS_button, CSS_url: None, waitTime: int = 3) -> list[dict]:
    # INIT 
    driver.get(BASE_URL)
    collected_headers = []
    raw_items: list[tuple[str, str, str]] = []

    time.sleep(waitTime)

    # ------------------------------------------------------------------
    # CASE 1: CSS_header is None → URL is included on the title element
    # ------------------------------------------------------------------
    if CSS_header == None:
        title_els = driver.find_elements(By.CSS_SELECTOR, CSS_title)
        date_els = driver.find_elements(By.CSS_SELECTOR, CSS_date)

        # iterate directly over elements to keep indices aligned
        for title_el, date_el in zip(title_els, date_els):
            try:
                title = title_el.text.strip()
                date = date_el.text.strip()
                url = title_el.get_attribute("href")
            except Exception:
                continue

            raw_items.append((url, title, date))

        if CSS_button != None:
            click_button(driver, CSS_button)

            title_els = driver.find_elements(By.CSS_SELECTOR, CSS_title)
            date_els = driver.find_elements(By.CSS_SELECTOR, CSS_date)

            # iterate directly over elements to keep indices aligned
            for title_el, date_el in zip(title_els, date_els):
                try:
                    title = title_el.text.strip()
                    date = date_el.text.strip()
                    url = title_el.get_attribute("href")
                except Exception:
                    continue
                
                raw_items.append((url, title, date))

    
    # ------------------------------------------------------------------
    # CASE 2: CSS_header selects elements that have the href (your original logic)
    # ------------------------------------------------------------------
    else:
        # Scraping 
        headers = driver.find_elements(By.CSS_SELECTOR, CSS_header)

        for head in headers:
            if CSS_url != None:
                    url_el = head.find_element(By.CSS_SELECTOR, CSS_url)
                    url = url_el.get_attribute('href')
            else: 
                url = head.get_attribute("href")

            if not url:
                continue

            try:
                title_el = head.find_element(By.CSS_SELECTOR, CSS_title)
                date_el = head.find_element(By.CSS_SELECTOR, CSS_date)
            except Exception:
                continue 

            title = title_el.text.strip()
            date = date_el.text.strip()

            # Clicking the 2024 button and scraping the title/date again
            if (not title or not date) and CSS_button:
                click_button(driver, CSS_button)

                try:
                    # same logic as your original: re-query inside the same header
                    title_el = head.find_element(By.CSS_SELECTOR, CSS_title)
                    date_el = head.find_element(By.CSS_SELECTOR, CSS_date)
                    title = title_el.text.strip()
                    date = date_el.text.strip()
                except Exception:
                    continue

            raw_items.append((url, title, date))

    # ------------------------------
    # Filter only 2024 and 2025 URLs
    # ------------------------------
    filtered_items = [
        (url, title, date)
        for (url, title, date) in raw_items
        if "2024" in date or "2025" in date
    ]

    print(f"Number of articles for 2024: {sum('2024' in date for _, _, date in filtered_items)}")
    print(f"Number of articles for 2025: {sum('2025' in date for _, _, date in filtered_items)}")

    # ------------------------------
    # Convert to list of dicts
    # ------------------------------
    collected_headers = [
        {"title": title, "date": date, "url": url}
        for (url, title, date) in filtered_items
    ]

    return collected_headers

# -------------------------------------------------------------------
# PHASE 2: ARTICLE SCRAPER
# -------------------------------------------------------------------
def scrape_article_content(driver, url, CSS_body, CSS_p: str = 'p'):
    # INIT
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    # Search for the article body
    article_body = wait.until(EC.presence_of_element_located((
        By.CSS_SELECTOR, CSS_body
    )))

    # Scrape the paragraphs 
    paras = article_body.find_elements(By.CSS_SELECTOR, CSS_p)

    content = "\n".join(p.text.strip() for p in paras if p.text.strip())

    # Sleep
    time.sleep(0.4)

    return content

# -------------------------------------------------------------------
# JSON saver
# -------------------------------------------------------------------
def save_json(records, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2,  ensure_ascii=False)


In [ ]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL, CSS_button):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info(driver, BASE_URL, "#pxl-content-main a", ".meta-date", ".jkit-postlist-title", CSS_button)
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                "[id^='pxl-post-'] [class*='text-editor']"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('DMX.V', "https://districtmetals.com/news-releases/", '#e-n-tab-title-1083811932 .e-n-tab-title-text')

Number of articles for 2024: 23
Number of articles for 2025: 30
number of articles scraped: 53
FAILED at url: https://districtmetals.com/news/2024/district-intersects-29-2-m-at-7-4-zneq-or-2-9-cueq-on-the-tomtebo-property/


## Doly varden silver Corp

In [ ]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL, CSS_button):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info(driver, BASE_URL, None, "time", ".elementor-page-title a", CSS_button)
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".elementor-widget-theme-post-content .elementor-widget-container"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('DV.V', "https://dollyvardensilver.com/news/", "#e-n-tab-title-2335627282")

Number of articles for 2024: 15
Number of articles for 2025: 29
number of articles scraped: 44


## Endurance Gold Corp

### 2024 and 2025

In [5]:

def scrape_egc_news():
    # Initialize the driver
    driver = webdriver.Chrome()  # or webdriver.Firefox()
    #driver.get("https://endurancegold.com/news-releases/2025/")
    driver.get("https://endurancegold.com/news-releases/2024/")
    
    wait = WebDriverWait(driver, 10)
    articles_data = []
    
    try:
        # You can find the CSS selectors by using SelectorGadget or inspecting the webpage
        article_links = driver.find_elements(By.CSS_SELECTOR, ".md\:pl-0 a")
        article_dates_elements = driver.find_elements(By.CSS_SELECTOR, "#content .px-2")
        article_urls = [link.get_attribute('href') for link in article_links]
        article_dates = [date.text for date in article_dates_elements]
        
        print(f"Found {len(article_urls)} articles to scrape")

        # Navigate to each article URL and extract data
        for idx, url in enumerate(article_urls):
            driver.get(url)
            time.sleep(2)  # Wait for page load
            
            try:
                title = driver.find_element(By.CSS_SELECTOR, ".uk-margin-small-bottom").text
                #content = driver.find_element(By.CSS_SELECTOR, ".category-78").text # for 2025
                content = driver.find_element(By.CSS_SELECTOR, ".mainContent").text # for 2024
                
                articles_data.append({
                    'title': title,
                    'date': article_dates[idx] if idx < len(article_dates) else "Unknown",
                    'content': content,
                    'url': url
                })
                
                print(f"Scraped: {title}")
            except Exception as e:
                print(f"Error scraping {url}: {e}")
        
        # Load existing data if file exists, otherwise start with empty list
        existing_data = []
        if os.path.exists('./Data/EDG.V_news.json'):
            try:
                with open('./Data/EDG.V_news.json', 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
                print(f"Loaded {len(existing_data)} existing articles")
            except json.JSONDecodeError:
                print("Warning: Existing JSON file was invalid, starting fresh")
        
        # Combine existing and new data (avoiding duplicates based on URL)
        existing_urls = {article['url'] for article in existing_data}
        new_articles = [article for article in articles_data if article['url'] not in existing_urls]
        
        combined_data = existing_data + new_articles
        
        # Save combined data to JSON
        with open('./Data/EDG.V_news.json', 'w', encoding='utf-8') as f:
            json.dump(combined_data, f, indent=2, ensure_ascii=False)
        
        print(f"Added {len(new_articles)} new articles. Total: {len(combined_data)} articles")
                
    finally:
        driver.quit()
    
    return articles_data

if __name__ == "__main__":
    articles = scrape_egc_news()
    print(f"Scraped {len(articles)} articles")

<>:12: SyntaxWarning: invalid escape sequence '\:'
<>:12: SyntaxWarning: invalid escape sequence '\:'
/var/folders/24/3ptrhbtn7x7cvnh_pyprvkrh0000gn/T/ipykernel_20434/2202795785.py:12: SyntaxWarning: invalid escape sequence '\:'
  article_links = driver.find_elements(By.CSS_SELECTOR, ".md\:pl-0 a")


Found 15 articles to scrape
Scraped: ENDURANCE ACCELERATES 100% OWNERSHIP OF THE ENTIRE RELIANCE GOLD PROJECT, B.C.
Scraped: ENDURANCE EXTENDS RELIANCE MINERALIZATION SIGNIFICANTLY AT DEPTH AND DISCOVERS NEW ZONE - NEW LOWER IMPERIAL ZONE WITH 7.18 GPT GOLD OVER 8.3 M INCLUDING 28.08 GPT GOLD OVER 1.7 M
Scraped: ENDURANCE REPORTS DRILL DISCOVERIES AT THE CROWN ZONE - 147.5 GPT GOLD OVER 1.0 METRE AND 7.61 GPT GOLD OVER 5.7 METRES
Scraped: ENDURANCE PROVIDES DRILLING UPDATE AT THE RELIANCE GOLD PROJECT, BC; SECOND DRILL ADDED - EAGLE ZONE EXPANDS AT DEPTH WITH 5.91 GPT GOLD OVER 3.0 M
Scraped: ENDURANCE ANNOUNCES FIRST 2024 DRILL RESULTS AT THE RELIANCE GOLD PROJECT, BC; INITIAL SIX HOLES EXPAND IMPERIAL ZONE WITH HIGH-GRADE GOLD MINERALIZATION
Scraped: ENDURANCE ANNOUNCES GRANT OF STOCK OPTIONS
Scraped: ENDURANCE PROVIDES RELIANCE GOLD PROJECT UPDATE; FIVE DEEP DRILL HOLES COMPLETED EXPANDING EXTENT OF IMPERIAL ZONE AT DEPTH
Scraped: ENDURANCE COMMENCES 10,000 M DRILLING PROGRAM AT THE

## Eloro Resources Ltd

In [ ]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info_page(driver, BASE_URL, ['2024', '2025'], ".date", ".title a")
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".news-entry"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('ELO.TO', "https://elororesources.com/news-and-media/news/")

Number of articles for 2024: 16
Number of articles for 2025: 0
Number of articles for 2024: 0
Number of articles for 2025: 24
number of articles scraped: 40


## Equinox Gold Corp

In een codeblock, the page and jaar aanpassen en dan lukt dit

In [8]:


def scrape_eqgc_news():
    # Initialize the driver
    driver = webdriver.Chrome()  # or webdriver.Firefox()
    # jaar aanpassen en pagina ook
    driver.get("https://www.equinoxgold.com/news/?query-1-page=3&cst&query-1-news-year=2025&query-1-search#fndry-query-1")
    
    wait = WebDriverWait(driver, 10)
    articles_data = []
    
    try:
        # You can find the CSS selectors by using SelectorGadget or inspecting the webpage
        article_links = driver.find_elements(By.CSS_SELECTOR, ".fndry-post-title__link")
        article_dates_elements = driver.find_elements(By.CSS_SELECTOR, ".fndry-text-h6Label14")
        article_urls = [link.get_attribute('href') for link in article_links]
        article_dates = [date.text for date in article_dates_elements]
        
        print(f"Found {len(article_urls)} articles to scrape")

        # Navigate to each article URL and extract data
        for idx, url in enumerate(article_urls):
            driver.get(url)
            time.sleep(2)  # Wait for page load
            
            try:
                title = driver.find_element(By.CSS_SELECTOR, ".has-text-align-left").text
                content = driver.find_element(By.CSS_SELECTOR, ".fndry-col--sm-12.fndry-pl--sm-2").text 
                
                articles_data.append({
                    'title': title,
                    'date': article_dates[idx] if idx < len(article_dates) else "Unknown",
                    'content': content,
                    'url': url
                })
                
                print(f"Scraped: {title}")
            except Exception as e:
                print(f"Error scraping {url}: {e}")
        
        # Load existing data if file exists, otherwise start with empty list
        existing_data = []
        if os.path.exists('./Data/eqgc_news.json'):
            try:
                with open('./Data/eqgc_news.json', 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
                print(f"Loaded {len(existing_data)} existing articles")
            except json.JSONDecodeError:
                print("Warning: Existing JSON file was invalid, starting fresh")
        
        # Combine existing and new data (avoiding duplicates based on URL)
        existing_urls = {article['url'] for article in existing_data}
        new_articles = [article for article in articles_data if article['url'] not in existing_urls]
        
        combined_data = existing_data + new_articles
        
        # Save combined data to JSON
        with open('./Data/eqgc_news.json', 'w', encoding='utf-8') as f:
            json.dump(combined_data, f, indent=2, ensure_ascii=False)
        
        print(f"Added {len(new_articles)} new articles. Total: {len(combined_data)} articles")
                
    finally:
        driver.quit()
    
    return articles_data

if __name__ == "__main__":
    articles = scrape_eqgc_news()
    print(f"Scraped {len(articles)} articles")

Found 5 articles to scrape
Scraped: Equinox Gold Reports Record Results for 2024 with 623,579 Ounces of Gold Sold, Revenue of $1.5 Billion and Operating Cash Flow of $430 Million
Scraped: Equinox Gold to Announce 2024 Financial and Operating Results and 2025 Guidance on February 19, 2025
Scraped: Equinox Gold Announces Filing of Fazenda Technical Report
Scraped: Equinox Gold Reports Record Quarterly and Annual Gold Production: Produced 213,960 Ounces of Gold in Q4 2024 and 621,870 Ounces of Gold for Full-year 2024
Scraped: Equinox Gold Extends Fazenda Mine Life by Seven Years with Updated Mineral Reserve and Mineral Resource Estimate
Loaded 51 existing articles
Added 5 new articles. Total: 56 articles
Scraped 5 articles


## Eskay mining

In [93]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info_page(driver, ".year", BASE_URL,['2024', '2025'], "[class*='first-column']", ".uk-width-expand\\@s", "a[href*='/news/']")
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".news-entry"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('ESK.V', "https://eskaymining.com/news/news-releases/")

Number of articles for 2024: 5
Number of articles for 2025: 0
Number of articles for 2024: 0
Number of articles for 2025: 8
number of articles scraped: 13


## Faraday Copper

2024 en 2025 apart scrapen, dus jaar aanpassen in code

In [21]:
def scrape_fc_news():
    # Initialize the driver
    driver = webdriver.Chrome()  # or webdriver.Firefox()
    driver.get("https://faradaycopper.com/news-releases/2025") # aanpassen foor 2024 en 2025
    
    wait = WebDriverWait(driver, 10)
    articles_data = []
    
    try:
        # You can find the CSS selectors by using SelectorGadget or inspecting the webpage
        article_links = driver.find_elements(By.CSS_SELECTOR, "#content .text-base")
        article_dates_elements = driver.find_elements(By.CSS_SELECTOR, ".px-2")
        article_urls = [link.get_attribute('href') for link in article_links]
        article_dates = [date.text for date in article_dates_elements]
        
        print(f"Found {len(article_urls)} articles to scrape")

        # Navigate to each article URL and extract data
        for idx, url in enumerate(article_urls):
            driver.get(url)
            time.sleep(2)  # Wait for page load
            
            try:
                title = driver.find_element(By.CSS_SELECTOR, ".uk-margin-small-bottom").text
                content = driver.find_element(By.CSS_SELECTOR, ".pr-6").text 
                
                articles_data.append({
                    'title': title,
                    'date': article_dates[idx] if idx < len(article_dates) else "Unknown",
                    'content': content,
                    'url': url
                })
                
                print(f"Scraped: {title}")
            except Exception as e:
                print(f"Error scraping {url}: {e}")
        
        # Load existing data if file exists, otherwise start with empty list
        existing_data = []
        if os.path.exists('./Data/fc_news.json'):
            try:
                with open('./Data/fc_news.json', 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
                print(f"Loaded {len(existing_data)} existing articles")
            except json.JSONDecodeError:
                print("Warning: Existing JSON file was invalid, starting fresh")
        
        # Combine existing and new data (avoiding duplicates based on URL)
        existing_urls = {article['url'] for article in existing_data}
        new_articles = [article for article in articles_data if article['url'] not in existing_urls]
        
        combined_data = existing_data + new_articles
        
        # Save combined data to JSON
        with open('./Data/fc_news.json', 'w', encoding='utf-8') as f:
            json.dump(combined_data, f, indent=2, ensure_ascii=False)
        
        print(f"Added {len(new_articles)} new articles. Total: {len(combined_data)} articles")
                
    finally:
        driver.quit()
    
    return articles_data

if __name__ == "__main__":
    articles = scrape_fc_news()
    print(f"Scraped {len(articles)} articles")

python(16372) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(16393) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Found 17 articles to scrape
Scraped: FARADAY COPPER REPORTS THIRD QUARTER 2025 FINANCIAL RESULTS
Scraped: FARADAY COPPER INITIATES LARGEST EVER DRILL PROGRAM AT THE COPPER CREEK PROJECT IN ARIZONA
Scraped: FARADAY COPPER REPORTS SECOND QUARTER 2025 FINANCIAL RESULTS
Scraped: FARADAY COPPER ANNOUNCES CLOSING OF C$49 MILLION FINANCING
Scraped: FARADAY COPPER UPSIZES FINANCING TO C$45 MILLION
Scraped: FARADAY COPPER ANNOUNCES C$30 MILLION FINANCING
Scraped: FARADAY COPPER RECEIVES BUREAU OF LAND MANAGEMENT APPROVAL FOR EXPLORATION PLAN OF OPERATIONS AT ITS COPPER CREEK PROJECT
Scraped: FARADAY COPPER EXPANDS NEAR-SURFACE MINERALIZATION IN THE AMERICAN EAGLE AREA AND IDENTIFIES EXPLORATION UPSIDE AT THE SUNRISE TREND
Scraped: FARADAY COPPER ANNOUNCES ANNUAL GENERAL MEETING VOTING RESULTS
Scraped: FARADAY COPPER INTERSECTS NEAR-SURFACE SUPERGENE COPPER MINERALIZATION AT THE GLOBE AND COPPER GIANT BRECCIAS, INCLUDING 10.08 METRES AT 3.62% COPPER AND 38.90 METRES AT 0.51% COPPER
Scraped: FARA

gelukt :)

## First Andes Silver Ltd

In [103]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL, CSS_button):
    try:
        driver = create_driver(headless=False)

        headers= collect_header_info(driver, BASE_URL, None, ".elementor-post-info__item--type-date", "#content a", CSS_button, None)
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".elementor-widget-theme-post-content .elementor-widget-container"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('FAS.V', "https://firstandes.com/news/", "#e-n-tab-title-2574086352 .e-n-tab-title-text")

Number of articles for 2024: 10
Number of articles for 2025: 17
number of articles scraped: 27


## Finex metals Ltd

Only data for 2025 available

In [6]:


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time
import json
import os

def scrape_FEM_news():
    # Initialize the driver
    driver = webdriver.Chrome()  # or webdriver.Firefox()
    driver.get("https://finexmetals.net/news/")
    
    wait = WebDriverWait(driver, 10)
    articles_data = []
    
    try:
        # You can find the CSS selectors by using SelectorGadget or inspecting the webpage
        article_links = driver.find_elements(By.CSS_SELECTOR, ".post__title")
        article_dates_elements = driver.find_elements(By.CSS_SELECTOR, ".post__date")
        article_urls = [link.get_attribute('href') for link in article_links]
        article_dates = [date.text for date in article_dates_elements]
        
        print(f"Found {len(article_urls)} articles to scrape")

        # Navigate to each article URL and extract data
        for idx, url in enumerate(article_urls):
            driver.get(url)
            time.sleep(2)  # Wait for page load
            
            try:
                title = driver.find_element(By.CSS_SELECTOR, "h2").text
                content = driver.find_element(By.CSS_SELECTOR, ".main-col").text # for 2024
                
                articles_data.append({
                    'title': title,
                    'date': article_dates[idx] if idx < len(article_dates) else "Unknown",
                    'content': content,
                    'url': url
                })
                
                print(f"Scraped: {title}")
            except Exception as e:
                print(f"Error scraping {url}: {e}")
        
        # Load existing data if file exists, otherwise start with empty list
        existing_data = []
        if os.path.exists('./Data/FEM_news.json'):
            try:
                with open('./Data/FEM_news.json', 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
                print(f"Loaded {len(existing_data)} existing articles")
            except json.JSONDecodeError:
                print("Warning: Existing JSON file was invalid, starting fresh")
        
        # Combine existing and new data (avoiding duplicates based on URL)
        existing_urls = {article['url'] for article in existing_data}
        new_articles = [article for article in articles_data if article['url'] not in existing_urls]
        
        combined_data = existing_data + new_articles
        
        # Save combined data to JSON
        with open('./Data/FEM_news.json', 'w', encoding='utf-8') as f:
            json.dump(combined_data, f, indent=2, ensure_ascii=False)
        
        print(f"Added {len(new_articles)} new articles. Total: {len(combined_data)} articles")
                
    finally:
        driver.quit()
    
    return articles_data

if __name__ == "__main__":
    articles = scrape_FEM_news()
    print(f"Scraped {len(articles)} articles")

Found 7 articles to scrape
Scraped: FINEX METALS REPORTS INITIAL DRILL RESULTS FROM RUOPPA EAST TARGET, FINLAND
Scraped: FINEX METALS COMPLETES INITIAL DRILLING ON RUOPPA GOLD PROJECT
Scraped: FINEX METALS COMMENCES DRILL PROGRAM AT RUOPPA GOLD PROJECT
Scraped: FINEX METALS PROVIDES EXPLORATION UPDATE ON RUOPPA GOLD PROJECT, FINLAND
Scraped: FINEX METALS BEGINS TRADING ON TSX VENTURE EXCHANGE AND ADVANCES 2025 EXPLORATION PROGRAM
Scraped: FINEX METALS LISTS ON THE TSX VENTURE EXCHANGE UNDER THE SYMBOL “FINX”
Scraped: FINEX METALS COMPLETES OVERSUBSCRIBED $4,356,000 PRIVATE PLACEMENT
Added 7 new articles. Total: 7 articles
Scraped 7 articles


## First Nordic Metals

In [109]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info(driver, BASE_URL, None, ".xpro-post-grid-meta-date", ".xpro-post-grid-meta-list+ a", None, None, waitTime = 30)
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".xpro-elementor-content"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('FNM.V', "https://firstnordicmetals.com/news/")

Number of articles for 2024: 24
Number of articles for 2025: 44
number of articles scraped: 68


## Foran Mining Corp

In [12]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL, CSS_button):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info(driver, BASE_URL, None , ".date-long", ".file-listing--title-wrapper", CSS_button, None) # ".file-listing--item-content"
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".col-lg-8"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('FOM.V', "https://foranmining.com/news-media/foran-news-releases/", '#tab-0-1 span')

Number of articles for 2024: 21
Number of articles for 2025: 28
number of articles scraped: 49


## Fury Gold Mines

In [ ]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL, yearlist):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info(driver, BASE_URL, None, '.post-date', '.post-title a', None, None)
        
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".user-content"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('FURY.TO', "https://furygoldmines.com/news-media/news-releases/", ['2024', '2025'])

Number of articles for 2024: 22
Number of articles for 2025: 26
number of articles scraped: 48


## Freegold Ventures Ltd

In [17]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL, CSS_button):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info(driver, BASE_URL, None , ".px-2", "#content a", CSS_button, None)
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".mainContent"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('FGV.V', "https://freegoldventures.com/news/2025/", '#nav-2024 .font-medium')

Number of articles for 2024: 14
Number of articles for 2025: 18
number of articles scraped: 32
FAILED at url: https://freegoldventures.com/site/assets/files/6455/fvl_12042025.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6449/fvl11202025.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6443/fvl11052025r.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6442/fvl10202025_frev1.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6440/fvl10072025f.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6435/fvl09042025.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6388/fvl_11112024_finalver1.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6380/fvl_10082024.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6365/fvl_09102024_final.pdf
FAILED at url: https://freegoldventures.com/site/assets/files/6363/fvl07222024.pdf


Paar sites waren pdf

## Fireweed Metals Corp

In [20]:
# -------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------
def main(ticker, BASE_URL, CSS_button):
    try:
        driver = create_driver(headless=False)

        headers = collect_header_info(driver, BASE_URL, None , "time", ".elementor-size-default a", CSS_button, None)
        print(f'number of articles scraped: {len(headers)}')

        # Article scraping 
        for head in headers:
            url = head['url']
            content = ""

            try:
                content = scrape_article_content(driver, 
                                                url,
                                                ".elementor-element-bd7b04e"
                )
            except:
                print(f'FAILED at url: {url}')

            head['content'] = content

        json_path = f'./Data/{ticker}_news.json'
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(headers, f, indent=2, ensure_ascii=False)
            
    # QUIT DRIVER WHEN FINISHED
    finally: 
        driver.quit()


if __name__ == "__main__":
    main('FWM.V', "https://fireweedmetals.com/news/", '#e-n-tab-title-675335872')

Number of articles for 2024: 23
Number of articles for 2025: 16
number of articles scraped: 39
